In [1]:
import pandas as pd
from urllib.parse import urlparse, urlunparse
from typing import Optional, Dict, Any, List


def _normalize_url(u: Optional[str], remove_query: bool=False, strip_www: bool=False, keep_scheme: bool=True) -> Optional[str]:
    """
    Normalize a URL string.
    - remove_query: drop query string (useful to ignore UTM params)
    - strip_www: remove leading 'www.' from netloc
    - keep_scheme: if False, strip scheme (useful if some URLs are stored without scheme)
    """
    if pd.isna(u):
        return None
    s = str(u).strip()
    if s == "":
        return None

    parsed = urlparse(s, scheme='' if keep_scheme else '')
    scheme = parsed.scheme.lower() if parsed.scheme else ''
    netloc = parsed.netloc.lower()
    if strip_www and netloc.startswith("www."):
        netloc = netloc[4:]
    path = parsed.path.rstrip('/')  # remove trailing slash
    query = '' if remove_query else parsed.query
    fragment = ''  # drop fragments

    # If no netloc but path looks like domain (e.g., example.com/path) urlparse may put domain in path.
    if not netloc and path:
        # naive fix: if path contains '.' before first '/', treat first token as netloc
        first = path.split('/', 1)[0]
        if '.' in first:
            # reconstruct
            rest = path[len(first):]
            netloc = first
            path = rest.rstrip('/')

    if keep_scheme:
        normalized = urlunparse((scheme, netloc, path, '', query, fragment))
    else:
        # omit scheme; produce netloc + path + optional query
        normalized = netloc + path
        if query:
            normalized += '?' + query

    # final cleanup
    normalized = normalized.strip()
    if normalized == '':
        return None
    return normalized



def _detect_url_column(df: pd.DataFrame, candidates: List[str]=None) -> Optional[str]:
    """
    Return the best guess column name for URL in df.
    Strategy: search for common substrings ('url', 'link', 'inputUrl', 'externalUrl') case-insensitive.
    If multiple matches, prefer exact 'url' then 'inputUrl' then first match.
    """
    if candidates is None:
        candidates = ['url', 'inputurl', 'externalurl', 'inputUrl', 'link', 'website', 'site']
    cols = df.columns.tolist()
    lower_cols = {c: c.lower() for c in cols}
    # prefer exact 'url' ignoring case
    for c, lc in lower_cols.items():
        if lc == 'url':
            return c
    # next prefer inputurl/externalurl
    for prefer in ['inputurl', 'externalurl', 'website', 'site', 'link']:
        for c, lc in lower_cols.items():
            if prefer in lc:
                return c
    # fallback: first column containing 'url' substring
    for c, lc in lower_cols.items():
        if 'url' in lc or 'link' in lc or 'site' in lc:
            return c
    return None


def merge_csvs_on_url(config: Dict[str, Any]) -> pd.DataFrame:
    """
    Merge two CSV files on URL using a configurable pipeline.

    Required keys in config:
      - left_path: path to first CSV (str)
      - right_path: path to second CSV (str)

    Optional keys (with defaults):
      - left_url_col: column name for URL in left CSV (if None -> auto-detect)
      - right_url_col: column name for URL in right CSV (if None -> auto-detect)
      - out_csv: path to save merged CSV (if provided)
      - out_excel: path to save Excel with sheets (matched, left_only, right_only) (if provided)
      - how: merge type 'inner'|'left'|'right'|'outer' (default 'inner')
      - remove_query: bool (default False) - whether to remove query string in normalization
      - strip_www: bool (default False) - whether to remove leading 'www.'
      - keep_scheme: bool (default True) - whether to keep scheme in normalization
      - drop_duplicates: bool (default False) - drop duplicates by normalized_url after merge
      - suffixes: tuple of 2 strings for overlapping columns (default ('_left','_right'))
      - verbose: bool (default True)
    Returns:
      merged DataFrame (pandas.DataFrame)
    """
    # --- defaults ---
    cfg = dict(
        left_url_col = None,
        right_url_col = None,
        out_csv = None,
        out_excel = None,
        how = 'inner',
        remove_query = False,
        strip_www = False,
        keep_scheme = True,
        drop_duplicates = False,
        suffixes = ('_left', '_right'),
        verbose = True,
    )
    # update defaults with provided config
    cfg.update(config)

    # read CSVs
    left = pd.read_csv(cfg['left_path'], dtype=str)
    right = pd.read_csv(cfg['right_path'], dtype=str)

    # detect url cols if None
    left_url_col = cfg['left_url_col'] or _detect_url_column(left)
    right_url_col = cfg['right_url_col'] or _detect_url_column(right)

    if cfg['verbose']:
        print(f"Left shape: {left.shape}, Right shape: {right.shape}")
        print(f"Left URL column: {left_url_col}, Right URL column: {right_url_col}")

    if left_url_col is None or right_url_col is None:
        raise ValueError("Could not detect URL column in one of the CSVs. Provide 'left_url_col' / 'right_url_col' in config.")

    # create normalized url columns
    left = left.copy()
    right = right.copy()
    left['_normalized_url'] = left[left_url_col].apply(lambda x: _normalize_url(x, remove_query=cfg['remove_query'], strip_www=cfg['strip_www'], keep_scheme=cfg['keep_scheme']))
    right['_normalized_url'] = right[right_url_col].apply(lambda x: _normalize_url(x, remove_query=cfg['remove_query'], strip_www=cfg['strip_www'], keep_scheme=cfg['keep_scheme']))

    if cfg['verbose']:
        n_left_missing = left['_normalized_url'].isna().sum()
        n_right_missing = right['_normalized_url'].isna().sum()
        print(f"Normalized - missing values: left={n_left_missing}, right={n_right_missing}")

    # perform merge
    merged = pd.merge(left, right, how=cfg['how'], on='_normalized_url', suffixes=tuple(cfg['suffixes']), indicator=True)

    # optionally drop duplicates by normalized_url
    if cfg['drop_duplicates']:
        merged = merged.drop_duplicates(subset=['_normalized_url'])

    # rename normalized column
    merged = merged.rename(columns={'_normalized_url': 'normalized_url'})

    # Save CSV if path provided
    if cfg['out_csv']:
        merged.to_csv(cfg['out_csv'], index=False)
        if cfg['verbose']:
            print(f"Wrote merged CSV to: {cfg['out_csv']}")

    # Optionally write Excel with sheets for matched / left_only / right_only
    if cfg['out_excel']:
        # matched = rows from merge where _merge == 'both'
        matched = merged[merged['_merge'] == 'both'].copy()
        left_only = merged[merged['_merge'] == 'left_only'].copy()
        right_only = merged[merged['_merge'] == 'right_only'].copy()

        with pd.ExcelWriter(cfg['out_excel'], engine='xlsxwriter') as writer:
            matched.to_excel(writer, sheet_name='matched', index=False)
            left_only.to_excel(writer, sheet_name='left_only', index=False)
            right_only.to_excel(writer, sheet_name='right_only', index=False)
        if cfg['verbose']:
            print(f"Wrote Excel report to: {cfg['out_excel']}")
            print(f"Counts -> matched: {len(matched)}, left_only: {len(left_only)}, right_only: {len(right_only)}")

    # final housekeeping: drop the merge indicator if you prefer
    # keep it for debugging unless user asked otherwise
    return merged


In [2]:
config = {
	'left_path': '/Users/utkarshumang/Desktop/rerun.csv',
	'right_path': 'cleaned_output_3.csv',
	'left_url_col': 'url',
	'right_url_col': 'url',
	'how': 'left',
	'remove_query': False,
	'strip_www': False,              
	'keep_scheme': True,            
	'drop_duplicates': True,
	'suffixes': ('_orig', '_sys'),
	'out_csv': 'merged_output.csv',
	'out_excel': None,
	'verbose': True,
}

merged_df = merge_csvs_on_url(config)
print("Merged rows:", len(merged_df))
display_cols = merged_df.columns.tolist()[:20]
display(merged_df.head()[display_cols])

Left shape: (3583, 6), Right shape: (3528, 17)
Left URL column: url, Right URL column: url
Normalized - missing values: left=0, right=0
Wrote merged CSV to: merged_output.csv
Merged rows: 3545


,description,keyword,network,proxyGroups/0,title,url_orig,normalized_url,inputUrl,ownerFullName,commentsCount,externalUrl,followersCount,followsCount,fullName,latestPosts/0/likesCount,latestPosts/1/likesCount,latestPosts/2/likesCount,latestPosts/3/likesCount,latestPosts/4/likesCount,likesCount
0,I now live in New York City and I started my o...,executive coach,Instagram,RESIDENTIAL,Reintroducing Maya Wofsy Wellness: Your Holist...,https://www.instagram.com/reel/DP4DpeqkQAl/,https://www.instagram.com/reel/DP4DpeqkQAl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,New York Flames age groups‼️Please share‼️ We ...,executive coach,Instagram,RESIDENTIAL,Introducing Firehawk Basketball's 2026 Coachin...,https://www.instagram.com/reel/DQKBrLIEV8u/,https://www.instagram.com/reel/DQKBrLIEV8u,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Sep 20, 2025—thepulsecreativelabs@gmail.com @t...",executive coach,Instagram,RESIDENTIAL,"The True Business of Basketball: Leadership, E...",https://www.instagram.com/reel/DO1iPR0k0Qw/-co...,https://www.instagram.com/reel/DO1iPR0k0Qw/-co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Email me with your guest ideas at otherpeoples...,executive coach,Instagram,RESIDENTIAL,Navigating Fear with Curiosity: Empowering Pro...,https://www.instagram.com/reel/DQZ7MxzD5gR/,https://www.instagram.com/reel/DQZ7MxzD5gR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Email: therelentlessroyal@gmail.com #YouthConf...,executive coach,Instagram,RESIDENTIAL,I appreciate your leadership and wisdom coach....,https://www.instagram.com/relentless_royal/p/D...,https://www.instagram.com/relentless_royal/p/D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
